# Orbital Propagation (Keplerian Model)

## Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

## Physical constants

In [2]:
MU = 398600.4418  # Earth's gravitational parameter [km^3/s^2]

## Convert mean motion to semi-major axis

𝑎
=
(
𝜇
(
2
𝜋
𝑛
86400
)
2
)
1
/
3
a=(
(
86400
2πn
	​

)
2
μ
	​

)
1/3

In [3]:
def mean_motion_to_semi_major_axis(n):
    """
    Convert mean motion (rev/day) to semi-major axis (km)
    """
    n_rad = n * 2 * np.pi / 86400  # rad/s
    a = (MU / n_rad**2)**(1/3)
    return a

## Solve Kepler’s Equation

In [4]:
def solve_kepler(M, e, tol=1e-8):
    """
    Solve Kepler's equation: M = E - e*sin(E)
    using Newton-Raphson iteration
    """
    E = M
    for _ in range(100):
        E = E - (E - e*np.sin(E) - M) / (1 - e*np.cos(E))
    return E

## Satellite propagation function

In [5]:
def propagate_satellite(sat, duration_minutes=90, step_seconds=60):
    """
    Propagate satellite position using a simple Keplerian model
    Returns latitude and longitude arrays
    """
    
    l2 = sat["l2"]
    
    # Extract orbital elements
    i = np.radians(float(l2[8:16]))              # inclination
    raan = np.radians(float(l2[17:25]))          # RAAN
    e = float("0." + l2[26:33].strip())          # eccentricity
    argp = np.radians(float(l2[34:42]))          # argument of perigee
    M0 = np.radians(float(l2[43:51]))            # mean anomaly
    n = float(l2[52:63])                         # mean motion
    
    # Semi-major axis
    a = mean_motion_to_semi_major_axis(n)
    
    lats = []
    lons = []
    
    for t in range(0, duration_minutes * 60, step_seconds):
        
        # Mean anomaly propagation
        M = M0 + (n * 2*np.pi / 86400) * t
        
        # Solve Kepler
        E = solve_kepler(M, e)
        
        # Position in orbital plane
        x = a * (np.cos(E) - e)
        y = a * np.sqrt(1 - e**2) * np.sin(E)
        
        # Rotate to ECI frame (simplified)
        X = x * np.cos(raan) - y * np.cos(i) * np.sin(raan)
        Y = x * np.sin(raan) + y * np.cos(i) * np.cos(raan)
        Z = y * np.sin(i)
        
        # Convert to latitude/longitude
        r = np.sqrt(X**2 + Y**2 + Z**2)
        lat = np.degrees(np.arcsin(Z / r))
        lon = np.degrees(np.arctan2(Y, X))
        
        lats.append(lat)
        lons.append(lon)
    
    return lats, lons

## Test with one satellite

In [6]:
sat = sats[0]

lats, lons = propagate_satellite(sat)

NameError: name 'sats' is not defined

## Plot ground track

In [7]:
plt.figure(figsize=(10,5))
plt.plot(lons, lats)
plt.xlabel("Longitude [deg]")
plt.ylabel("Latitude [deg]")
plt.title(f"Ground Track - {sat['name']}")
plt.grid()
plt.show()

NameError: name 'lons' is not defined

<Figure size 1000x500 with 0 Axes>